# Notebook 01: Limpieza e Integración de Datos (Data Preprocessing)
**Proyecto:** Sistema Inteligente de Triaje para Soporte IT
**Objetivo de este Notebook:** Transformar los datos brutos (raw) del ecosistema "Help Desk Tickets" en un Dataset Maestro consolidado (`processed`). 

**Alineación con los objetivos del TFG:**
1. **Reconstrucción del contexto:** Agrupar los mensajes fragmentados del cliente (`reporter`) para recrear el "correo de entrada" real.
2. **Definición del Ground Truth (Verdad Absoluta):** Vincular el texto con la prioridad real asignada por la empresa (`issue_priority`).
3. **Validación Experta:** Incorporar las notas de evaluación del Manager para tener contexto de resolución en casos límite.

In [8]:
import os
import pandas as pd
import numpy as np
import re
import warnings

warnings.filterwarnings('ignore')

# 1. Definición de la estructura de carpetas local
# Usamos rutas relativas para que el proyecto sea portable
BASE_DIR = os.path.dirname(os.getcwd()) # Sube un nivel desde 'notebooks'
DATA_RAW = os.path.join(BASE_DIR, "data", "raw")
DATA_PROCESSED = os.path.join(BASE_DIR, "data", "processed")

# Crear carpetas si no existen
os.makedirs(DATA_RAW, exist_ok=True)
os.makedirs(DATA_PROCESSED, exist_ok=True)

# 2. Definición de archivos
PATH_ISSUES = os.path.join(DATA_RAW, "issues.csv")
PATH_TEXTOS = os.path.join(DATA_RAW, "sample_utterances.csv")
PATH_EXCEL = os.path.join(DATA_RAW, "issues_snapshot_sample.xlsx")
PATH_OUTPUT = os.path.join(DATA_PROCESSED, "dataset_validacion_tfg.csv")

print(f"✅ Entorno listo.\n📂 Datos brutos en: {DATA_RAW}\n📂 Salida procesada en: {DATA_PROCESSED}")

✅ Entorno listo.
📂 Datos brutos en: c:\Users\barco\OneDrive\Documentos\GitHub\it-mail-service-desk\data\raw
📂 Salida procesada en: c:\Users\barco\OneDrive\Documentos\GitHub\it-mail-service-desk\data\processed


In [9]:
print("⏳ Cargando datos...")

try:
    df_issues = pd.read_csv(PATH_ISSUES)
    df_textos = pd.read_csv(PATH_TEXTOS)
    # Nota: Para leer Excel local necesitas: pip install openpyxl
    df_sample = pd.read_excel(PATH_EXCEL)

    print(f"📊 Tickets (issues): {df_issues.shape[0]}")
    print(f"📊 Mensajes (textos): {df_textos.shape[0]}")
    print(f"📊 Evaluaciones Manager: {df_sample.shape[0]}")
except Exception as e:
    print(f"❌ Error al cargar archivos: {e}\n⚠️ Asegúrate de tener los archivos en la carpeta data/raw/")

⏳ Cargando datos...
📊 Tickets (issues): 66691
📊 Mensajes (textos): 30104
📊 Evaluaciones Manager: 747


In [10]:
# 1. Filtrar mensajes del cliente (reporter)
df_clientes = df_textos[df_textos['author_role'] == 'reporter'].copy()

# 2. Ordenar y limpiar
df_clientes = df_clientes.sort_values(by=['issueid', 'comment_seq', 'utr_seq'])
df_clientes = df_clientes.dropna(subset=['actionbody'])

# 3. Quedarnos solo con el comment_seq == 0 (Apertura del ticket)
df_primer_contacto = df_clientes[df_clientes['comment_seq'] == 0]

# 4. Agrupar mensajes fragmentados
df_textos_agrupados = df_primer_contacto.groupby('issueid')['actionbody'].apply(lambda x: ' \n '.join(x)).reset_index()
df_textos_agrupados.rename(columns={'actionbody': 'texto_cliente_completo'}, inplace=True)

print(f"📧 Tickets con correo de apertura reconstruido: {df_textos_agrupados.shape[0]}")

📧 Tickets con correo de apertura reconstruido: 355


In [11]:
def limpiar_texto(texto):
    if not isinstance(texto, str): return ""
    
    texto = texto.lower()
    # Anonimización de placeholders
    texto = re.sub(r'ph_ip_address', '[IP]', texto)
    texto = re.sub(r'ph_user', '[USER]', texto)
    texto = re.sub(r'ph_technical', '[TECH_TERM]', texto)
    
    # Limpieza de espacios
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df_textos_agrupados['texto_limpio'] = df_textos_agrupados['texto_cliente_completo'].apply(limpiar_texto)
print("✨ Texto normalizado para el LLM.")

✨ Texto normalizado para el LLM.


In [12]:
# Unir con información de prioridad y tiempos
df_master = pd.merge(df_textos_agrupados, 
                     df_issues[['id', 'issue_priority', 'issue_type', 'wf_total_time']], 
                     left_on='issueid', right_on='id', how='inner')

# Unir con notas del Manager para auditoría
if 'id' in df_sample.columns:
    df_master = pd.merge(df_master, df_sample[['id', 'Notes', 'Q1', 'Q2', 'Q3']], on='id', how='left')

df_master.drop(columns=['id'], inplace=True)
print(f"🔗 Dataset integrado. Tamaño final: {df_master.shape}")

🔗 Dataset integrado. Tamaño final: (355, 10)


In [13]:
def asignar_urgencia_tfg(row):
    # Clase 1: Urgencia explícita
    if row['issue_priority'] in ['High', 'Highest', 'Blocker']:
        return 1
    
    # Clase Especial: Auditoría de Medium
    # Si es Medium pero el Manager detectó un problema en Q1/Q2 o hay notas, lo subimos a 1
    # Esto reduce el "ruido" que detectamos en los tests de la IA
    if row['issue_priority'] == 'Medium':
        # Si existe nota del manager, asumimos que requirió atención (Urgente para el TFG)
        if pd.notnull(row['Notes']):
            return 1
            
    return 0

df_master['es_urgente_real'] = df_master.apply(asignar_urgencia_tfg, axis=1)

print("📊 Distribución de es_urgente_real:")
print(df_master['es_urgente_real'].value_counts(normalize=True) * 100)

📊 Distribución de es_urgente_real:
es_urgente_real
1    84.788732
0    15.211268
Name: proportion, dtype: float64


In [14]:
df_master.to_csv(PATH_OUTPUT, index=False, encoding='utf-8')
print(f"💾 Dataset guardado en: {PATH_OUTPUT}")
print("🚀 ¡Todo listo para ejecutar el script de clasificación local!")

💾 Dataset guardado en: c:\Users\barco\OneDrive\Documentos\GitHub\it-mail-service-desk\data\processed\dataset_validacion_tfg.csv
🚀 ¡Todo listo para ejecutar el script de clasificación local!
